In [1]:
import json

# First, let's assume you have your JSON data in a file named 'results.json'
# Let's load the data from that file
with open('cruft/results-macrof1.json') as f:
    results = json.load(f)

commontop = []
# Now, let's iterate through each dataset in the results and print the F1 scores ordered from best to worst
first = True
for dataset, classifiers in results.items():
    print(f"Results for {dataset}:")
    # Create a list of tuples where each tuple is (classifier_name, f1_score)
    f1_scores = [(classifier, data['f1']) for classifier, data in classifiers.items()]
    # Sort the list by f1 score in descending order
    f1_scores_sorted = sorted(f1_scores, key=lambda x: x[1], reverse=True)
    
    currenttop = set()
    # Print the sorted f1 scores
    for classifier, f1 in f1_scores_sorted[:10]:
        print(f"{classifier}: {f1}")
        if first:
            commontop.append(classifier)
        else:
            currenttop.add(classifier)
    
    if not first:
        commontop = list(set(commontop).intersection(currenttop))

    # Add a newline for better separation between datasets
    print()
    first=False

print(commontop)

Results for tweet_eval_hate:
AdaBoostClassifier: 0.5033077727875881
DecisionTreeClassifier: 0.5029633338418698
HistGradientBoostingClassifier: 0.49966140324162606
Perceptron: 0.29682313892840206

Results for ade_corpus_v2:
HistGradientBoostingClassifier: 0.6412217646005475
Perceptron: 0.6272
AdaBoostClassifier: 0.5923041359934563
DecisionTreeClassifier: 0.5898958535134261

Results for overruling:
AdaBoostClassifier: 0.8301327614871215
HistGradientBoostingClassifier: 0.8283711286548723
DecisionTreeClassifier: 0.7729842668518243
Perceptron: 0.33690744920993226

Results for banking_77:
DecisionTreeClassifier: 0.059278936748120445
Perceptron: 0.04284171143891354
HistGradientBoostingClassifier: 0.040223143369647586
AdaBoostClassifier: 0.0037856292768176816

['Perceptron', 'HistGradientBoostingClassifier', 'DecisionTreeClassifier', 'AdaBoostClassifier']


In [2]:
import json
import pandas as pd

# Load the data from the JSON file
with open('cruft/results-macrof1.json') as f:
    results = json.load(f)

# Initialize a dictionary to store the F1 scores for the table
table_data = {}

# Process each dataset and extract the F1 scores
for dataset, classifiers in results.items():
    for classifier, data in classifiers.items():
        if classifier not in table_data:
            table_data[classifier] = {}
        table_data[classifier][dataset] = data['f1']

# Convert the dictionary to a DataFrame for better display
df = pd.DataFrame(table_data).T

# Display the DataFrame
print(df)

# Save the DataFrame to a CSV file if needed
df.to_csv('results_table.csv')


                                tweet_eval_hate  ade_corpus_v2  overruling  \
AdaBoostClassifier                     0.503308       0.592304    0.830133   
DecisionTreeClassifier                 0.502963       0.589896    0.772984   
HistGradientBoostingClassifier         0.499661       0.641222    0.828371   
Perceptron                             0.296823       0.627200    0.336907   

                                banking_77  
AdaBoostClassifier                0.003786  
DecisionTreeClassifier            0.059279  
HistGradientBoostingClassifier    0.040223  
Perceptron                        0.042842  


In [2]:
import pandas as pd
from datasets import load_dataset, load_from_disk
from sklearn.metrics import f1_score

from app.constants import *


pairs = [
    # ('13/predictions/ade_corpus_v2', 'cruft/raft_ade_corpus_v2'),
    # ('14/predictions/tweet_eval_hate', 'cruft/raft_tweet_eval_hate'),
    # ('15/predictions/overruling', 'cruft/raft_overruling'),
    # ('16/predictions/banking_77', 'cruft/raft_banking_77'),


    # 13/predictions/ade_corpus_v2, F1 Score: 0.6016
    # 14/predictions/tweet_eval_hate, F1 Score: 0.4437
    # 15/predictions/overruling, F1 Score: 0.8356
    # 16/predictions/banking_77, F1 Score: 0.0208


    # ('25/predictions/ade_corpus_v2', 'cruft/raft_ade_corpus_v2'),
    # ('25/predictions/tweet_eval_hate', 'cruft/raft_tweet_eval_hate'),
    # ('25/predictions/overruling', 'cruft/raft_overruling'),
    # ('25/predictions/banking_77', 'cruft/raft_banking_77'),


    # 25/predictions/ade_corpus_v2, F1 Score: 0.4463
    # 25/predictions/tweet_eval_hate, F1 Score: 0.3662
    # 25/predictions/overruling, F1 Score: 0.3297
    # 25/predictions/banking_77, F1 Score: 0.0004


    ('2/predictions/ade_corpus_v2', 'cruft/raft_ade_corpus_v2'),
    ('2/predictions/tweet_eval_hate', 'cruft/raft_tweet_eval_hate'),
    ('2/predictions/overruling', 'cruft/raft_overruling'),
    ('2/predictions/banking_77', 'cruft/raft_banking_77'),
]


runs = [
    ('gpt2 distilled', '2'),
    ('gpt2', '3'),
    ('gpt2-large', '5'),

]


for model, run_id in runs:

    for d in RAFT_DATASETS:
        csv_file = f'../raft-baselines/src/results/make_predictions/{run_id}/predictions/{d}/predictions.csv'
        df = pd.read_csv(csv_file)

        # Map the labels from the CSV to 0 and 1

        # Load the HuggingFace dataset
        dataset = load_from_disk(f'cruft/raft_{d}')

        # Convert the HuggingFace dataset to a DataFrame for easy manipulation
        hf_df = pd.DataFrame(dataset['test'])

        # Merge the two DataFrames on the ID column
        merged_df = pd.merge(df, hf_df, on='ID', suffixes=('_csv', '_hf'))

        merged_df['Label_csv'] = merged_df['Label_csv'].map(lambda x: dataset['test'].features['Label'].str2int(x))
        merged_df['Label_hf'] = merged_df['Label_hf'] - 1
        merged_df['Label_csv'] = merged_df['Label_csv'] - 1


        # Calculate the F1 score
        f1 = f1_score(merged_df['Label_hf'], merged_df['Label_csv'], average='macro')
        print(f'{model}, {d} F1 Score: {f1:.4f}')


gpt2 distilled, tweet_eval_hate F1 Score: 0.1169
gpt2 distilled, ade_corpus_v2 F1 Score: 0.0042
gpt2 distilled, overruling F1 Score: 0.0597
gpt2 distilled, banking_77 F1 Score: 0.0098
gpt2, tweet_eval_hate F1 Score: 0.0027
gpt2, ade_corpus_v2 F1 Score: 0.0308
gpt2, overruling F1 Score: 0.0417
gpt2, banking_77 F1 Score: 0.0200
gpt2-large, tweet_eval_hate F1 Score: 0.0419
gpt2-large, ade_corpus_v2 F1 Score: 0.2474
gpt2-large, overruling F1 Score: 0.0194
gpt2-large, banking_77 F1 Score: 0.0109
